# SP-02 Mean-Reversion Policy — `02_mean_reversion_policy.ipynb`

Second specialized policy of the Personalized AI Trading Agent. Output: `mean_reversion_policy.pt` plus config, schema, preprocessing, metrics and equity curve.

**Research prototype for backtesting and paper trading only. Not investment advice.**

## What this notebook does
1. Downloads daily OHLCV for a configurable liquid-equity universe and a benchmark (yfinance).
2. Builds **causal** features (rolling statistics only look backwards).
3. Builds labels. **Default = proxy labels from an explicit mean-reversion rule (strategy imitation).** No genuine trader actions are used unless you supply a CSV (Section 5).
4. Trains a GRU/LSTM classifier with the action space `0=HOLD, 1=BUY, 2=SELL, 3=EXIT`.
5. Evaluates with chronological splits, purged label horizons, a next-bar-open backtest with costs, and compares against (a) the benchmark and (b) the plain rule with no learning.
6. Exports artifacts, runs an inference demo and a Meta-RL compatibility report.

## Read this before interpreting any number
* **Proxy-label accuracy measures how well the network reproduces the rule, not trading skill and not human behavior.** Entry labels are filtered by *future* reversal evidence (an oracle the live policy never sees), so the network must predict that evidence from past data. That is the learning problem.
* Backtest profitability is **not** implied by the labels. Mean reversion can lose money, especially in trends. Whatever the backtest shows is what it shows.
* Default universe is a list of **currently** large, liquid stocks → **survivorship bias** inflates results. Prices are split/dividend adjusted by Yahoo with full-sample factors.
* Daily bars, simple cost model, no volume-participation limit, no intraday stops. Treat results as indicative only.

## 1. Installation

In [ ]:
%pip install -q yfinance

## 2. Imports, device, reproducibility

In [ ]:
import os, sys, json, math, time, random, pickle, hashlib, warnings, platform, zlib
from dataclasses import dataclass, field, asdict
from typing import Optional, Dict, List, Tuple, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("python", platform.python_version(), "| torch", torch.__version__, "| pandas", pd.__version__,
      "| numpy", np.__version__, "| sklearn", sklearn.__version__)
print("device:", DEVICE, (torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "(CPU fallback)"))

## 3. Configuration

Everything tunable lives in `Config`. Action mapping is shared by all five policies:
`0 = HOLD, 1 = BUY (open long), 2 = SELL (open short), 3 = EXIT (close any open position)`.

Validity rules (enforced in the label generator, inside the network via logit masking, and in the backtest engine):

| Position | HOLD | BUY | SELL | EXIT |
|---|---|---|---|---|
| flat | ok | ok | ok **only if `allow_short`** | invalid |
| long / short | ok | invalid | invalid | ok |

No pyramiding and no flipping in one step (EXIT first, then re-enter on a later bar).

In [ ]:
ACTIONS = {0: "HOLD", 1: "BUY", 2: "SELL", 3: "EXIT"}
ACTION_IDS = {v: k for k, v in ACTIONS.items()}

DEFAULT_TICKERS = ("AAPL", "MSFT", "AMZN", "GOOGL", "META", "NVDA", "JPM", "XOM", "JNJ", "PG",
                   "V", "UNH", "HD", "KO", "PEP", "WMT", "BAC", "CVX", "MRK", "COST")

@dataclass
class Config:
    # ---- data ----
    data_source: str = "yfinance"            # "yfinance" | "synthetic_smoke_test" (pipeline test only, results meaningless)
    tickers: tuple = DEFAULT_TICKERS
    benchmark: str = "SPY"
    start: str = "2010-01-01"
    end: Optional[str] = None                # None = today
    min_history_frac: float = 0.95           # drop tickers with less history than this fraction of benchmark
    # ---- sequence ----
    seq_len: int = 32
    max_hold_bars: int = 15                  # time stop in the rule and normaliser for holding_bars
    # ---- labels ----
    label_source: str = "proxy_rule_mean_reversion"   # "proxy_rule_mean_reversion" | "genuine_trader_actions"
    trader_actions_path: Optional[str] = None           # CSV: date,ticker,action (see Section 5)
    trader_actions_fill_hold: bool = True               # dates absent from the CSV are treated as HOLD
    z_entry: float = 1.5                     # |z-score(20)| needed for an entry candidate
    rsi_oversold: float = 40.0
    rsi_overbought: float = 60.0
    adx_max: float = 30.0                    # skip candidates in strong trends
    exit_z: float = 0.25                     # exit when z-score returns within +-exit_z of the mean
    label_horizon: int = 10                  # bars after entry in which reversal evidence must appear
    reversion_frac: float = 0.5              # fraction of the gap to the 20-bar mean that must be recovered
    stop_atr_mult: float = 2.0               # adverse move (in ATR) that invalidates the setup / triggers stop exit
    min_target_gain: float = 0.003           # minimum expected gain (fraction) for a candidate to be worth labeling BUY/SELL
    # ---- model ----
    rnn_type: str = "GRU"                    # "GRU" | "LSTM"
    hidden_dim: int = 64
    num_layers: int = 2
    dense_dim: int = 64
    dropout: float = 0.2
    # ---- training ----
    train_frac: float = 0.60
    val_frac: float = 0.20                   # test = remainder
    batch_size: int = 256
    epochs: int = 60
    lr: float = 1e-3
    weight_decay: float = 1e-4
    patience: int = 8
    grad_clip: float = 1.0
    class_weight_power: float = 0.5          # 1.0 = full inverse-frequency, 0 = none
    seed: int = 42
    # ---- backtest ----
    initial_capital: float = 100_000.0
    fee_bps: float = 1.0                     # per side
    slippage_bps: float = 5.0                # per side, adverse to the trade
    borrow_bps_annual: float = 50.0          # on short market value
    allow_short: bool = True                 # long/short restriction (False = long-only; SELL is then masked)
    max_position_frac: float = 0.10          # max notional per position as a fraction of equity
    max_gross_exposure: float = 1.0          # gross exposure / equity cap (no margin leverage by default)
    min_entry_confidence: float = 0.0        # optional gate on BUY/SELL max-probability
    min_trade_notional: float = 100.0
    risk_free_annual: float = 0.0
    # ---- trader profile (constants unless you supply real profile data) ----
    profile_supplied: bool = False
    profile: dict = field(default_factory=lambda: {
        "risk_tolerance": 0.5,               # 0..1
        "max_position_size": 0.10,           # fraction of equity, 0..1
        "max_drawdown_tolerance": 0.20,      # fraction, 0..1
        "preferred_holding_days": 10.0,
    })
    artifact_dir: str = "artifacts"

cfg = Config()
set_seed(cfg.seed)
os.makedirs(cfg.artifact_dir, exist_ok=True)

LABEL_DISCLOSURE = (
    "LABELS ARE PROXY LABELS from an explicit mean-reversion rule (strategy imitation). "
    "They are NOT genuine trader behavior and results do not show learned human decision-making."
    if cfg.label_source == "proxy_rule_mean_reversion" else
    "LABELS ARE GENUINE TRADER ACTIONS supplied via CSV; quality depends on that data.")
print(LABEL_DISCLOSURE)

## 4. Data download

Daily OHLCV via `yfinance` (`auto_adjust=True`). Tickers are aligned onto a common trading calendar (intersection of dates) so the whole universe can be handled as one `(tickers x time x features)` panel. A synthetic generator exists **only** to smoke-test the pipeline offline; it is disabled by default and any result obtained with it is meaningless.

In [ ]:
def download_ohlcv(ticker, start, end):
    import yfinance as yf
    last_err = None
    for attempt in range(3):
        try:
            df = yf.Ticker(ticker).history(start=start, end=end, auto_adjust=True, actions=False)
            if df is not None and len(df) > 0:
                break
        except Exception as e:
            last_err = e
        time.sleep(2)
    else:
        raise RuntimeError(f"Download failed for {ticker}: {last_err}")
    df = df[["Open", "High", "Low", "Close", "Volume"]].copy()
    df.index = pd.to_datetime(df.index).tz_localize(None).normalize()
    df = df[~df.index.duplicated(keep="last")].sort_index()
    df = df[(df[["Open", "High", "Low", "Close"]] > 0).all(axis=1)]
    return df

def make_synthetic(ticker, index):
    # PIPELINE SMOKE TEST ONLY: random walk + AR(1) mean-reverting component. Not market data.
    rng = np.random.default_rng(zlib.crc32(ticker.encode()))
    n = len(index)
    ou = np.zeros(n)
    for i in range(1, n):
        ou[i] = 0.92 * ou[i - 1] + 0.012 * rng.standard_normal()
    logp = np.log(100) + np.cumsum(0.0003 + 0.008 * rng.standard_normal(n)) + ou
    c = np.exp(logp)
    o = np.r_[c[0], c[:-1]] * (1 + 0.002 * rng.standard_normal(n))
    h = np.maximum(o, c) * (1 + np.abs(0.004 * rng.standard_normal(n)))
    l = np.minimum(o, c) * (1 - np.abs(0.004 * rng.standard_normal(n)))
    v = np.exp(15 + 0.3 * rng.standard_normal(n))
    return pd.DataFrame({"Open": o, "High": h, "Low": l, "Close": c, "Volume": v}, index=index)

raw = {}
if cfg.data_source == "yfinance":
    for tk in list(cfg.tickers) + [cfg.benchmark]:
        try:
            raw[tk] = download_ohlcv(tk, cfg.start, cfg.end)
            print(f"{tk:6s} {len(raw[tk]):5d} bars  {raw[tk].index[0].date()} -> {raw[tk].index[-1].date()}")
        except Exception as e:
            print(f"{tk:6s} FAILED: {e}")
    if cfg.benchmark not in raw:
        raise RuntimeError("Benchmark download failed; cannot continue.")
elif cfg.data_source == "synthetic_smoke_test":
    print("WARNING: synthetic data - pipeline smoke test only. Results are meaningless.")
    idx = pd.bdate_range(cfg.start, periods=3000)
    for tk in list(cfg.tickers) + [cfg.benchmark]:
        raw[tk] = make_synthetic(tk, idx)
else:
    raise ValueError(cfg.data_source)

# keep tickers with enough history, then align on the common calendar
bench = raw[cfg.benchmark]
kept = [tk for tk in cfg.tickers if tk in raw
        and len(raw[tk]) >= cfg.min_history_frac * len(bench)
        and raw[tk].index[0] <= bench.index[0] + pd.Timedelta(days=10)]
dropped = [tk for tk in cfg.tickers if tk not in kept]
if dropped:
    print("dropped (missing/short history):", dropped)
common = bench.index
for tk in kept:
    common = common.intersection(raw[tk].index)
common = common.sort_values()
assert len(kept) >= 3 and len(common) > 500, "Not enough aligned data."
DATES = common
NAMES = kept
print(f"\nUniverse: {len(NAMES)} tickers | aligned bars: {len(DATES)} | {DATES[0].date()} -> {DATES[-1].date()}")

## 5. Feature engineering (causal)

Every rolling statistic at bar *t* uses bars `<= t` only. Features are computed per ticker on the aligned calendar; benchmark features are market context.

**Feature groups (the common schema, 29 inputs per timestep)**

* **Market (22):** log returns (1/5/10/20/60 bars), deviation from SMA20, rolling return std, Bollinger position and width, z-score of price vs rolling mean (20, 60), RSI(14), ATR% (14), realized volatility, volume change and relative volume, ADX(14), regression-slope trend strength, Kaufman efficiency ratio, SMA20/SMA60 ratio, benchmark 20-bar return and volatility.
* **Position state (3):** `position` (-1/0/+1), `unrealized_pnl` (fraction of entry price, signed by direction), `holding_bars`. State at bar *t* = position held at the close of *t*; an order decided at the close of *t* fills at the open of *t+1*.
* **Trader profile / risk (4):** `risk_tolerance`, `max_position_size`, `max_drawdown_tolerance`, `preferred_holding_days`. **Constants** unless `profile_supplied=True` with real data, so the network cannot learn profile dependence from this dataset. They are in the schema for cross-policy compatibility.

In [ ]:
MARKET_FEATURES = [
    "ret_1", "ret_5", "ret_10", "ret_20", "ret_60",
    "sma_dev_20", "roll_std_20", "bb_pos_20", "bb_width_20", "zscore_20", "zscore_60",
    "rsi_14", "atr_14_pct", "rv_20", "vol_chg_1", "vol_ratio_20",
    "adx_14", "trend_slope_20", "eff_ratio_20", "sma_ratio_20_60",
    "bench_ret_20", "bench_rv_20",
]
STATE_FEATURES = ["position", "unrealized_pnl", "holding_bars"]
PROFILE_FEATURES = ["risk_tolerance", "max_position_size", "max_drawdown_tolerance", "preferred_holding_days"]
FEATURE_NAMES = MARKET_FEATURES + STATE_FEATURES + PROFILE_FEATURES
F = len(FEATURE_NAMES)
FIDX = {n: i for i, n in enumerate(FEATURE_NAMES)}
MARKET_IDX = [FIDX[n] for n in MARKET_FEATURES]
STATE_IDX = [FIDX[n] for n in STATE_FEATURES]
I_POS, I_UP, I_HOLD = FIDX["position"], FIDX["unrealized_pnl"], FIDX["holding_bars"]

def rsi_wilder(close, n=14):
    d = close.diff()
    up, dn = d.clip(lower=0), -d.clip(upper=0)
    au = up.ewm(alpha=1 / n, adjust=False, min_periods=n).mean()
    ad = dn.ewm(alpha=1 / n, adjust=False, min_periods=n).mean()
    rsi = 100 - 100 / (1 + au / ad.replace(0, np.nan))
    rsi[(ad == 0) & (au > 0)] = 100.0
    return rsi

def true_range(h, l, c):
    pc = c.shift(1)
    return pd.concat([h - l, (h - pc).abs(), (l - pc).abs()], axis=1).max(axis=1)

def adx_wilder(h, l, c, n=14):
    up_move, dn_move = h.diff(), -l.diff()
    plus_dm = pd.Series(np.where((up_move > dn_move) & (up_move > 0), up_move, 0.0), index=c.index)
    minus_dm = pd.Series(np.where((dn_move > up_move) & (dn_move > 0), dn_move, 0.0), index=c.index)
    atr = true_range(h, l, c).ewm(alpha=1 / n, adjust=False, min_periods=n).mean()
    pdi = 100 * plus_dm.ewm(alpha=1 / n, adjust=False, min_periods=n).mean() / atr
    mdi = 100 * minus_dm.ewm(alpha=1 / n, adjust=False, min_periods=n).mean() / atr
    dx = 100 * (pdi - mdi).abs() / (pdi + mdi).replace(0, np.nan)
    return dx.ewm(alpha=1 / n, adjust=False, min_periods=n).mean()

_x = np.arange(20) - 9.5
_w = _x / np.sum(_x ** 2)

def compute_features(df, bench_close):
    # All operations are backward-looking (rolling / ewm / diff / shift with positive lags).
    o, h, l, c, v = df["Open"], df["High"], df["Low"], df["Close"], df["Volume"]
    logc = np.log(c)
    r1 = logc.diff()
    f = pd.DataFrame(index=df.index)
    for k in (1, 5, 10, 20, 60):
        f[f"ret_{k}"] = logc.diff(k)
    sma20, sd20 = c.rolling(20).mean(), c.rolling(20).std(ddof=0)
    sma60, sd60 = c.rolling(60).mean(), c.rolling(60).std(ddof=0)
    f["sma_dev_20"] = c / sma20 - 1
    f["roll_std_20"] = r1.rolling(20).std(ddof=0)
    upper, lower = sma20 + 2 * sd20, sma20 - 2 * sd20
    f["bb_pos_20"] = (c - lower) / (upper - lower).replace(0, np.nan)
    f["bb_width_20"] = (upper - lower) / sma20
    f["zscore_20"] = (c - sma20) / sd20.replace(0, np.nan)
    f["zscore_60"] = (c - sma60) / sd60.replace(0, np.nan)
    f["rsi_14"] = rsi_wilder(c)
    atr = true_range(h, l, c).ewm(alpha=1 / 14, adjust=False, min_periods=14).mean()
    f["atr_14_pct"] = atr / c
    f["rv_20"] = f["roll_std_20"] * np.sqrt(252)
    vol = v.replace(0, np.nan)
    f["vol_chg_1"] = np.log(vol / vol.shift(1)).clip(-3, 3)
    f["vol_ratio_20"] = np.log(vol / vol.rolling(20).mean()).clip(-3, 3)
    f["adx_14"] = adx_wilder(h, l, c)
    slope = logc.rolling(20).apply(lambda y: float(np.dot(_w, y)), raw=True)
    f["trend_slope_20"] = (slope / f["roll_std_20"].replace(0, np.nan)).clip(-5, 5)
    f["eff_ratio_20"] = (c - c.shift(20)).abs() / c.diff().abs().rolling(20).sum().replace(0, np.nan)
    f["sma_ratio_20_60"] = sma20 / sma60 - 1
    bc = bench_close.reindex(df.index).ffill()
    f["bench_ret_20"] = np.log(bc / bc.shift(20))
    f["bench_rv_20"] = np.log(bc).diff().rolling(20).std(ddof=0) * np.sqrt(252)
    f = f.replace([np.inf, -np.inf], np.nan)
    first_valid = f.dropna().index[0]
    f = f.ffill()                                   # causal fill of isolated gaps after warm-up
    aux = pd.DataFrame({"sma_20": sma20, "atr_14": atr}, index=df.index)
    return f[MARKET_FEATURES], aux, first_valid

# ---- build the aligned panel (N tickers x T bars x F features) ----
N, T = len(NAMES), len(DATES)
O = np.zeros((N, T)); Hh = np.zeros((N, T)); L_ = np.zeros((N, T)); C = np.zeros((N, T))
PR = np.zeros((N, T, F))                       # raw (unscaled) feature panel
SMA20 = np.zeros((N, T)); ATR = np.zeros((N, T))
FV = np.zeros(N, dtype=int)
bench_close = bench["Close"]
for n, tk in enumerate(NAMES):
    d = raw[tk].loc[DATES]
    feat, aux, fvd = compute_features(d, bench_close)
    O[n], Hh[n], L_[n], C[n] = d["Open"].values, d["High"].values, d["Low"].values, d["Close"].values
    PR[n][:, MARKET_IDX] = feat.values
    SMA20[n], ATR[n] = aux["sma_20"].values, aux["atr_14"].values
    FV[n] = DATES.get_loc(fvd)
FV_GLOBAL = int(FV.max())
B = bench["Close"].reindex(DATES).ffill().values        # benchmark close on the aligned calendar
for j, name in enumerate(PROFILE_FEATURES):
    PR[:, :, FIDX[name]] = cfg.profile[name]
assert np.isfinite(PR[:, FV_GLOBAL:, :][:, :, MARKET_IDX]).all(), "NaN/inf in features after warm-up"
print(f"panel {PR.shape} | features per step: {F} | first fully valid bar: {DATES[FV_GLOBAL].date()}")
pd.DataFrame(PR[0][FV_GLOBAL:, MARKET_IDX], columns=MARKET_FEATURES, index=DATES[FV_GLOBAL:]).describe().T.round(4)

## 6. Label construction

### Decision-time information vs. later outcomes
At the close of bar *t* the decision may use only features at `<= t` (and the current position state). The order fills at the **open of bar t+1**.

### Proxy rule (default): documented mean-reversion strategy
*Entry candidate* (flat position, decision at bar *t*):
* **Long:** `zscore_20 <= -z_entry`, `RSI <= rsi_oversold`, `ADX <= adx_max`.
* **Short** (only if `allow_short`): mirror image.
* The ADX cap deliberately excludes strong trends, where oversold/overbought readings often persist.

*Reversal evidence (uses the FUTURE, labels only):* entry price `p0 = open[t+1]`. Reference level = 20-bar SMA at *t* (frozen). Target = `p0 + reversion_frac * (SMA20_t - p0)`. Stop = `p0 -/+ stop_atr_mult * ATR_t`. The candidate is labeled **BUY/SELL only if the target is touched within `label_horizon` bars before the stop** (stop is checked first inside a bar, i.e. conservatively). Otherwise it is labeled **HOLD**: this is how the model learns that *not every oversold market recovers*. Candidates whose expected gain is below `min_target_gain` are HOLD.

*Exit (causal, no future info):* **EXIT** when the z-score returns within `exit_z` of the mean, or the position loses more than `stop_atr_mult * ATR%` (current ATR), or `holding_bars >= max_hold_bars`.

*Position state* is produced by rolling the rule forward in time, so state features are exactly what the rule's own trades would have produced.

### Leakage controls
* Entry labels look ahead up to `label_horizon + 1` bars. Samples within `label_horizon + 2` bars before each split boundary are **purged**.
* Bars within `label_horizon` of the end of data have unknown outcomes and are unlabeled.
* Overlapping input windows are only shared *within* a split (inputs are past features); the purge removes label overlap *across* splits.

### Genuine trader actions (optional)
Set `cfg.label_source = "genuine_trader_actions"` and `cfg.trader_actions_path` to a CSV with columns `date,ticker,action` (action = `HOLD/BUY/SELL/EXIT` or `0..3`). Position state is then replayed from those actions; invalid actions (e.g. BUY while long) are converted to HOLD and counted. No such data is bundled, so the default run is **proxy-label only**.

In [ ]:
def proxy_labels_one(o, h, l, c, z, rsi, adx, atr, atr_pct, sma, fv, cfg):
    T_ = len(c)
    y = np.full(T_, -1, dtype=np.int8)
    pos = np.zeros(T_); up = np.zeros(T_); hold = np.zeros(T_)
    p, entry_px, entry_idx = 0, np.nan, -1
    Hn = cfg.label_horizon
    for t in range(fv, T_ - 1):
        if p != 0:                                   # state at close of t
            pos[t] = p; hold[t] = t - entry_idx + 1; up[t] = p * (c[t] / entry_px - 1.0)
        if p == 0:
            cand = 0
            if z[t] <= -cfg.z_entry and rsi[t] <= cfg.rsi_oversold and adx[t] <= cfg.adx_max:
                cand = 1
            elif cfg.allow_short and z[t] >= cfg.z_entry and rsi[t] >= cfg.rsi_overbought and adx[t] <= cfg.adx_max:
                cand = -1
            if cand == 0:
                y[t] = 0; continue
            e = t + 1
            if e + Hn > T_:
                y[t] = -1; continue                  # outcome not observable -> unlabeled
            p0 = o[e]
            gap = (sma[t] - p0) / p0
            if cand * gap * cfg.reversion_frac < cfg.min_target_gain:
                y[t] = 0; continue
            target = p0 + cfg.reversion_frac * (sma[t] - p0)
            stop = p0 - cand * cfg.stop_atr_mult * atr[t]
            ok = False
            for k in range(e, e + Hn):               # FUTURE bars: used for the label only
                if cand == 1:
                    if l[k] <= stop: break
                    if h[k] >= target: ok = True; break
                else:
                    if h[k] >= stop: break
                    if l[k] <= target: ok = True; break
            if ok:
                y[t] = 1 if cand == 1 else 2
                p, entry_px, entry_idx = cand, o[e], e
            else:
                y[t] = 0
        else:
            reverted = (z[t] >= -cfg.exit_z) if p == 1 else (z[t] <= cfg.exit_z)
            stopped = up[t] <= -cfg.stop_atr_mult * atr_pct[t]
            timeout = hold[t] >= cfg.max_hold_bars
            if reverted or stopped or timeout:
                y[t] = 3; p, entry_px, entry_idx = 0, np.nan, -1
            else:
                y[t] = 0
    if p != 0:
        t = T_ - 1; pos[t] = p; hold[t] = t - entry_idx + 1; up[t] = p * (c[t] / entry_px - 1.0)
    return y, pos, up, hold

def replay_state(y_in, o, c, fv, cfg):
    # Replays supplied (genuine) actions to obtain position state; invalid actions become HOLD.
    T_ = len(c); y = y_in.copy()
    pos = np.zeros(T_); up = np.zeros(T_); hold = np.zeros(T_)
    p, entry_px, entry_idx, invalid = 0, np.nan, -1, 0
    for t in range(fv, T_ - 1):
        if p != 0:
            pos[t] = p; hold[t] = t - entry_idx + 1; up[t] = p * (c[t] / entry_px - 1.0)
        a = int(y[t])
        if a < 0: continue
        if a == 1 and p == 0: p, entry_px, entry_idx = 1, o[t + 1], t + 1
        elif a == 2 and p == 0 and cfg.allow_short: p, entry_px, entry_idx = -1, o[t + 1], t + 1
        elif a == 3 and p != 0: p, entry_px, entry_idx = 0, np.nan, -1
        elif a != 0: y[t] = 0; invalid += 1
    return y, pos, up, hold, invalid

Y = np.full((N, T), -1, dtype=np.int8)
if cfg.label_source == "proxy_rule_mean_reversion":
    for n in range(N):
        z = PR[n][:, FIDX["zscore_20"]]; rsi = PR[n][:, FIDX["rsi_14"]]; adx = PR[n][:, FIDX["adx_14"]]
        atrp = PR[n][:, FIDX["atr_14_pct"]]
        y, pos, up, hold = proxy_labels_one(O[n], Hh[n], L_[n], C[n], z, rsi, adx, ATR[n], atrp, SMA20[n], FV[n], cfg)
        Y[n] = y; PR[n][:, I_POS] = pos; PR[n][:, I_UP] = up; PR[n][:, I_HOLD] = hold
elif cfg.label_source == "genuine_trader_actions":
    assert cfg.trader_actions_path, "Set cfg.trader_actions_path"
    ta = pd.read_csv(cfg.trader_actions_path, parse_dates=["date"])
    ta["action"] = ta["action"].map(lambda a: ACTION_IDS[a.upper()] if isinstance(a, str) and not a.isdigit() else int(a))
    total_invalid = 0
    for n, tk in enumerate(NAMES):
        y0 = np.full(T, -1, dtype=np.int8)
        if cfg.trader_actions_fill_hold: y0[FV[n]:T - 1] = 0
        sub = ta[ta["ticker"] == tk]
        for d_, a_ in zip(sub["date"], sub["action"]):
            if d_ in DATES: y0[DATES.get_loc(d_)] = a_
        y, pos, up, hold, inv = replay_state(y0, O[n], C[n], FV[n], cfg)
        total_invalid += inv
        Y[n] = y; PR[n][:, I_POS] = pos; PR[n][:, I_UP] = up; PR[n][:, I_HOLD] = hold
    print("invalid trader actions converted to HOLD:", total_invalid)
else:
    raise ValueError(cfg.label_source)

lab = Y[Y >= 0]
print(LABEL_DISCLOSURE)
print("overall label counts:", {ACTIONS[k]: int((lab == k).sum()) for k in range(4)})

## 7. Chronological split, purge, train-only preprocessing

Splits are by **calendar index** (same dates for every ticker): first 60 % train, next 20 % validation, last 20 % test. Samples closer than `label_horizon + 2` bars to the end of train or validation are purged. The scaler is fit **only on training-period rows**; features are clipped at ±5 sigma after scaling. State and profile columns use fixed, data-independent transforms.

In [ ]:
class Preprocessor:
    def __init__(self, feature_names, market_features, clip=5.0, upnl_scale=0.10, max_hold=15):
        self.feature_names = list(feature_names)
        self.market_idx = [self.feature_names.index(n) for n in market_features]
        self.i_up = self.feature_names.index("unrealized_pnl")
        self.i_hold = self.feature_names.index("holding_bars")
        self.clip, self.upnl_scale, self.max_hold = float(clip), float(upnl_scale), float(max_hold)
        self.scaler, self.mean_, self.scale_ = None, None, None

    def fit(self, rows):                                   # rows: (n, F) raw, TRAIN ONLY
        self.scaler = StandardScaler().fit(rows[:, self.market_idx])
        self.mean_ = self.scaler.mean_.copy()
        self.scale_ = np.where(self.scaler.scale_ > 1e-12, self.scaler.scale_, 1.0)
        return self

    def scale_upnl(self, x): return np.clip(np.asarray(x, dtype=np.float64) / self.upnl_scale, -self.clip, self.clip)
    def scale_hold(self, x): return np.clip(np.asarray(x, dtype=np.float64) / self.max_hold, 0.0, 3.0)

    def transform(self, X):                                # X: (..., F) raw -> float32 model input
        X = np.asarray(X, dtype=np.float64)
        out = X.copy()
        out[..., self.market_idx] = np.clip((X[..., self.market_idx] - self.mean_) / self.scale_, -self.clip, self.clip)
        out[..., self.i_up] = self.scale_upnl(X[..., self.i_up])
        out[..., self.i_hold] = self.scale_hold(X[..., self.i_hold])
        return out.astype(np.float32)                      # position and profile pass through unchanged

    def to_dict(self):
        return {"feature_names": self.feature_names, "market_idx": self.market_idx,
                "mean": [float(v) for v in self.mean_], "scale": [float(v) for v in self.scale_],
                "clip": self.clip, "upnl_scale": self.upnl_scale, "max_hold": self.max_hold}

    @classmethod
    def from_dict(cls, d):
        names = d["feature_names"]
        p = cls(names, [names[i] for i in d["market_idx"]], d["clip"], d["upnl_scale"], d["max_hold"])
        p.mean_, p.scale_ = np.array(d["mean"]), np.array(d["scale"])
        return p

SEQ = cfg.seq_len
PURGE = cfg.label_horizon + 2
I_TRAIN_END, I_VAL_END = int(T * cfg.train_frac), int(T * (cfg.train_frac + cfg.val_frac))
tgrid = np.arange(T)[None, :].repeat(N, axis=0)
win_ok = tgrid >= FV_GLOBAL + SEQ - 1
has_label = (Y >= 0) & win_ok
train_mask = has_label & (tgrid < I_TRAIN_END - PURGE)
val_mask = has_label & (tgrid >= I_TRAIN_END) & (tgrid < I_VAL_END - PURGE)
test_mask = has_label & (tgrid >= I_VAL_END)
print(f"train: {DATES[0].date()} -> {DATES[I_TRAIN_END-1].date()} | purge {PURGE} bars")
print(f"val  : {DATES[I_TRAIN_END].date()} -> {DATES[I_VAL_END-1].date()} | purge {PURGE} bars")
print(f"test : {DATES[I_VAL_END].date()} -> {DATES[-1].date()}")

PRE = Preprocessor(FEATURE_NAMES, MARKET_FEATURES, clip=5.0, upnl_scale=0.10, max_hold=cfg.max_hold_bars)
PRE.fit(PR[:, FV_GLOBAL:I_TRAIN_END, :].reshape(-1, F))            # train period only
PX = PRE.transform(PR)                                              # (N, T, F) float32 model-input panel

def idx_from(mask): return np.nonzero(mask)
tr_tk, tr_t = idx_from(train_mask); va_tk, va_t = idx_from(val_mask); te_tk, te_t = idx_from(test_mask)
y_tr, y_va, y_te = Y[tr_tk, tr_t].astype(int), Y[va_tk, va_t].astype(int), Y[te_tk, te_t].astype(int)
cnt = pd.DataFrame({s: np.bincount(y, minlength=4) for s, y in zip(["train", "val", "test"], [y_tr, y_va, y_te])},
                   index=[ACTIONS[k] for k in range(4)])
print("\nsamples per split and class:\n", cnt)
assert len(y_tr) > 200 and len(y_va) > 50 and len(y_te) > 50, "Too few samples; widen the universe/date range or relax rule thresholds."
cnt.plot(kind="bar", figsize=(7, 3), title="Class distribution per split (log scale)", logy=True); plt.tight_layout(); plt.show()

## 8. Model

`MeanReversionNet`: **GRU/LSTM sequence encoder** → LayerNorm → **dense representation layer** (GELU, dropout) → **4-way action head**. The validity mask (from the *current* `position` input) is applied to the logits inside the network, so invalid actions have probability exactly zero in training, evaluation and inference. `confidence = max probability`.

In [ ]:
NEG = -1e9

class MeanReversionNet(nn.Module):
    def __init__(self, n_features, hidden_dim=64, num_layers=2, dense_dim=64, dropout=0.2,
                 rnn_type="GRU", pos_idx=I_POS, allow_short=True, n_actions=4):
        super().__init__()
        rnn_cls = {"GRU": nn.GRU, "LSTM": nn.LSTM}[rnn_type]
        self.encoder = rnn_cls(n_features, hidden_dim, num_layers=num_layers, batch_first=True,
                               dropout=dropout if num_layers > 1 else 0.0)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dense = nn.Sequential(nn.Linear(hidden_dim, dense_dim), nn.GELU(), nn.Dropout(dropout))
        self.head = nn.Linear(dense_dim, n_actions)
        self.pos_idx, self.allow_short = pos_idx, allow_short

    def valid_mask(self, x):
        flat = x[:, -1, self.pos_idx].abs() < 0.5
        m = torch.ones(x.shape[0], 4, dtype=torch.bool, device=x.device)
        m[:, 1] = flat
        m[:, 2] = flat & self.allow_short
        m[:, 3] = ~flat
        return m

    def representation(self, x):
        out, _ = self.encoder(x)
        return self.dense(self.norm(out[:, -1, :]))

    def forward(self, x):
        logits = self.head(self.representation(x))
        return logits.masked_fill(~self.valid_mask(x), NEG)

MODEL_KWARGS = dict(n_features=F, hidden_dim=cfg.hidden_dim, num_layers=cfg.num_layers, dense_dim=cfg.dense_dim,
                    dropout=cfg.dropout, rnn_type=cfg.rnn_type, pos_idx=I_POS, allow_short=cfg.allow_short)
_m = MeanReversionNet(**MODEL_KWARGS)
print(_m)
print("parameters:", sum(p.numel() for p in _m.parameters()))
del _m

## 9. Training

* Cross-entropy with **class weights** computed from the *training* split only: `(N / (K * n_c)) ** class_weight_power`. Classes absent from training (e.g. SELL with `allow_short=False`) get weight 0.
* AdamW, gradient clipping, **early stopping on validation weighted loss**, best weights restored.
* Windows are gathered on the fly from the scaled panel, so overlapping windows cost no extra memory.
* Note: weighting raises recall of rare entry actions at the cost of precision; the backtest later shows what that does in practice.

In [ ]:
def gather(P, tk, tt, L):
    rows = tt[:, None] + np.arange(-L + 1, 1)[None, :]
    return torch.from_numpy(P[tk[:, None], rows])           # (B, L, F)

@torch.no_grad()
def predict_probs(model, P, tk, tt, L, bs=2048):
    model.eval(); out = []
    for i in range(0, len(tk), bs):
        x = gather(P, tk[i:i + bs], tt[i:i + bs], L).to(DEVICE)
        out.append(torch.softmax(model(x), dim=-1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, 4))

def class_weights(y, power):
    counts = np.bincount(y, minlength=4).astype(float)
    present = counts > 0
    w = np.zeros(4)
    w[present] = (counts.sum() / (present.sum() * counts[present])) ** power
    w[present] /= w[present].mean()
    return w

set_seed(cfg.seed)
model = MeanReversionNet(**MODEL_KWARGS).to(DEVICE)
W = class_weights(y_tr, cfg.class_weight_power)
print("class weights:", {a: round(float(w), 3) for a, w in zip(ACTIONS.values(), W)})
criterion = nn.CrossEntropyLoss(weight=torch.tensor(W, dtype=torch.float32, device=DEVICE))
opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)
rng = np.random.default_rng(cfg.seed)

@torch.no_grad()
def eval_loss(tk, tt, y):
    model.eval(); tot, wsum, preds = 0.0, 0.0, []
    for i in range(0, len(tk), 2048):
        x = gather(PX, tk[i:i + 2048], tt[i:i + 2048], SEQ).to(DEVICE)
        yy = torch.from_numpy(y[i:i + 2048]).to(DEVICE)
        lg = model(x); tot += criterion(lg, yy).item() * len(yy); wsum += len(yy)
        preds.append(lg.argmax(-1).cpu().numpy())
    pr = np.concatenate(preds)
    present = sorted(set(y.tolist()))
    return tot / wsum, f1_score(y, pr, labels=present, average="macro", zero_division=0), pr

hist = {"train_loss": [], "val_loss": [], "val_macro_f1": [], "lr": []}
best, best_state, bad = np.inf, None, 0
for epoch in range(1, cfg.epochs + 1):
    model.train(); perm = rng.permutation(len(tr_tk)); run, seen = 0.0, 0
    for i in range(0, len(perm), cfg.batch_size):
        b = perm[i:i + cfg.batch_size]
        x = gather(PX, tr_tk[b], tr_t[b], SEQ).to(DEVICE)
        yy = torch.from_numpy(y_tr[b]).to(DEVICE)
        opt.zero_grad(); loss = criterion(model(x), yy); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip); opt.step()
        run += loss.item() * len(b); seen += len(b)
    vl, vf, _ = eval_loss(va_tk, va_t, y_va); sched.step(vl)
    hist["train_loss"].append(run / seen); hist["val_loss"].append(vl); hist["val_macro_f1"].append(vf)
    hist["lr"].append(opt.param_groups[0]["lr"])
    flag = ""
    if vl < best - 1e-5:
        best, bad, flag = vl, 0, " *"
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        bad += 1
    print(f"epoch {epoch:3d} | train {run/seen:.4f} | val {vl:.4f} | val macro-F1 {vf:.3f}{flag}")
    if bad >= cfg.patience:
        print(f"early stopping at epoch {epoch}"); break
model.load_state_dict(best_state); model.eval()
BEST_EPOCH = int(np.argmin(hist["val_loss"])) + 1
print("best epoch:", BEST_EPOCH, "| best val loss:", round(best, 4))

## 10. Learning curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist["train_loss"], label="train"); ax[0].plot(hist["val_loss"], label="validation")
ax[0].axvline(BEST_EPOCH - 1, color="grey", ls="--", lw=0.8); ax[0].set_title("Weighted cross-entropy"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(hist["val_macro_f1"], color="tab:green"); ax[1].set_title("Validation macro-F1 (present classes)"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 11. Classification evaluation (teacher-forced)

Metrics below score the network against the **labels** on the held-out test period, with the position state taken from the rule's own path (teacher forcing). Because labels are proxy labels, **this measures rule imitation, not profitability and not human behavior.** Class-imbalance matters: read per-class precision/recall, not accuracy.

In [ ]:
def cls_metrics(tk, tt, y):
    probs = predict_probs(model, PX, tk, tt, SEQ)
    pred = probs.argmax(1)
    rep = classification_report(y, pred, labels=[0, 1, 2, 3], target_names=list(ACTIONS.values()), zero_division=0, output_dict=True)
    cm = confusion_matrix(y, pred, labels=[0, 1, 2, 3])
    present = sorted(set(y.tolist()))
    return dict(accuracy=float(accuracy_score(y, pred)),
                macro_f1_present=float(f1_score(y, pred, labels=present, average="macro", zero_division=0)),
                always_hold_accuracy=float((y == 0).mean()),
                report=rep, confusion_matrix=cm.tolist()), probs, pred, cm

cls_val, _, _, _ = cls_metrics(va_tk, va_t, y_va)
cls_test, te_probs, te_pred, cm_test = cls_metrics(te_tk, te_t, y_te)
print(LABEL_DISCLOSURE, "\n")
print(f"TEST accuracy {cls_test['accuracy']:.3f} (always-HOLD baseline {cls_test['always_hold_accuracy']:.3f}) | "
      f"macro-F1 over present classes {cls_test['macro_f1_present']:.3f}\n")
print(classification_report(y_te, te_pred, labels=[0, 1, 2, 3], target_names=list(ACTIONS.values()), zero_division=0))

fig, ax = plt.subplots(figsize=(4.6, 4))
ax.imshow(cm_test, cmap="Blues")
ax.set_xticks(range(4)); ax.set_yticks(range(4)); ax.set_xticklabels(ACTIONS.values()); ax.set_yticklabels(ACTIONS.values())
ax.set_xlabel("predicted"); ax.set_ylabel("label (proxy)"); ax.set_title("Test confusion matrix")
for i in range(4):
    for j in range(4):
        ax.text(j, i, int(cm_test[i, j]), ha="center", va="center", color="white" if cm_test[i, j] > cm_test.max() / 2 else "black")
plt.tight_layout(); plt.show()

x_chk = gather(PX, te_tk[:5000], te_t[:5000], SEQ).to(DEVICE)
with torch.no_grad():
    vm = model.valid_mask(x_chk).cpu().numpy(); pr_chk = torch.softmax(model(x_chk), -1).cpu().numpy()
print("max probability mass on invalid actions (test sample):", float((pr_chk * (~vm)).max()))

## 12. Backtest

Portfolio simulation over the whole universe, daily bars:

* Decision at the **close of bar t** → order fills at the **open of bar t+1** (never the same bar).
* Position state fed to the model is the backtest's **own** live state (not the rule's), so the model trades on its own path.
* **Costs:** `fee_bps` per side plus `slippage_bps` per side, adverse to the trade; `borrow_bps_annual` accrues on short value.
* **Capital:** a long needs enough cash; per-position notional <= `max_position_frac * equity`; gross exposure <= `max_gross_exposure * equity`. Competing entries on the same bar are served in descending confidence.
* **Restrictions:** `allow_short=False` makes SELL impossible. BUY/SELL while in a position and EXIT while flat are blocked (counted).
* Open positions are liquidated (with costs) on the last bar.
* **Benchmark:** buy-and-hold of the benchmark over the same dates (no costs).
* **Rule baseline:** the same rule *without* the future reversal filter and without any learning, run through the same engine, so you can see whether the network adds anything beyond the rule it imitates.

The model has **no stop-loss besides what it learned** (EXIT behaviour); thresholds such as `min_entry_confidence` are not tuned on the test set.

In [ ]:
D = dict(O=O, H=Hh, L=L_, C=C, B=B, names=NAMES)

def run_backtest(policy_fn, start, end, cfg):
    N_, T_ = C.shape
    PXbt = PX.copy(); PXbt[:, :, STATE_IDX] = 0.0                  # model sees its OWN position state only
    fee, slip, borrow_d = cfg.fee_bps / 1e4, cfg.slippage_bps / 1e4, cfg.borrow_bps_annual / 1e4 / 252
    cash = float(cfg.initial_capital)
    shares = np.zeros(N_); entry_px = np.full(N_, np.nan); entry_idx = np.full(N_, -1); entry_fee = np.zeros(N_)
    pending, eq, expo, trades, traded, blocked = [], [], [], [], 0.0, 0

    def close_pos(n, t, px, forced):
        nonlocal cash, traded
        sgn = np.sign(shares[n]); fill = px * (1 - slip * sgn)
        notional = abs(shares[n]) * fill; cost = notional * fee
        cash += shares[n] * fill - cost; traded += notional
        pnl = shares[n] * (fill - entry_px[n]) - cost - entry_fee[n]
        trades.append(dict(ticker=NAMES[n], side="LONG" if sgn > 0 else "SHORT", entry_bar=int(entry_idx[n]), exit_bar=int(t),
                           entry_date=str(DATES[int(entry_idx[n])].date()), exit_date=str(DATES[t].date()),
                           entry_px=float(entry_px[n]), exit_px=float(fill), shares=float(shares[n]), pnl=float(pnl),
                           ret=float(pnl / (abs(shares[n]) * entry_px[n])), bars_held=int(t - entry_idx[n]), forced_close=forced))
        shares[n] = 0.0; entry_px[n] = np.nan; entry_idx[n] = -1; entry_fee[n] = 0.0

    for t in range(start, end + 1):
        # 1) execute orders decided at the previous close, at this bar's open
        for (n, a, cf) in sorted(pending, key=lambda z: (0 if z[1] == 3 else 1, -z[2])):
            if a == 3 and shares[n] != 0:
                close_pos(n, t, O[n, t], False)
            elif a in (1, 2) and shares[n] == 0:
                sgn = 1 if a == 1 else -1
                fill = O[n, t] * (1 + slip * sgn)
                eq_open = cash + (shares * O[:, t]).sum()
                gross = np.abs(shares * O[:, t]).sum()
                notional = min(cfg.max_position_frac * eq_open, cfg.max_gross_exposure * eq_open - gross)
                if sgn == 1: notional = min(notional, cash / (1 + fee))
                if notional < cfg.min_trade_notional: continue
                sh = notional / fill; cost = notional * fee
                cash -= sgn * sh * fill + cost
                shares[n] = sgn * sh; entry_px[n] = fill; entry_idx[n] = t; entry_fee[n] = cost; traded += notional
        pending = []
        # 2) mark to market at the close
        sv = shares * C[:, t]
        cash -= borrow_d * (-sv[sv < 0].sum())
        eq_t = cash + sv.sum(); eq.append(eq_t); expo.append(np.abs(sv).sum() / eq_t)
        # 3) own state at the close of t
        pos = np.sign(shares)
        hold = np.where(pos != 0, t - entry_idx + 1, 0).astype(float)
        with np.errstate(invalid="ignore"):
            up = np.where(pos != 0, pos * (C[:, t] / entry_px - 1.0), 0.0)
        PXbt[:, t, I_POS] = pos; PXbt[:, t, I_UP] = PRE.scale_upnl(up); PXbt[:, t, I_HOLD] = PRE.scale_hold(hold)
        # 4) decide (fills next bar)
        if t < end:
            acts, conf = policy_fn(t, pos, up, hold, PXbt)
            for n in np.nonzero(acts > 0)[0]:
                a = int(acts[n])
                ok = (a == 1 and pos[n] == 0) or (a == 2 and pos[n] == 0 and cfg.allow_short) or (a == 3 and pos[n] != 0)
                if ok: pending.append((int(n), a, float(conf[n])))
                else: blocked += 1
    for n in np.nonzero(shares != 0)[0]:
        close_pos(n, end, C[n, end], True)
    eq[-1] = cash
    return dict(equity=np.array(eq), trades=pd.DataFrame(trades), traded=traded, blocked=blocked, exposure=float(np.mean(expo)))

def perf(eq, rf_annual):
    r = eq[1:] / eq[:-1] - 1.0
    years = max(len(r) / 252.0, 1e-9)
    sd = r.std(ddof=1) if len(r) > 2 else np.nan
    dd = eq / np.maximum.accumulate(eq) - 1.0
    return dict(cumulative_return=float(eq[-1] / eq[0] - 1), cagr=float((eq[-1] / eq[0]) ** (1 / years) - 1),
                annual_vol=float(sd * np.sqrt(252)), sharpe=float((r.mean() - rf_annual / 252) / sd * np.sqrt(252)) if sd and sd > 0 else float("nan"),
                max_drawdown=float(dd.min()))

def summarize(res, start, end, cfg):
    eq = res["equity"]; tr = res["trades"]; years = len(eq) / 252.0
    m = perf(eq, cfg.risk_free_annual)
    m.update(trade_count=int(len(tr)), win_rate=float((tr["pnl"] > 0).mean()) if len(tr) else float("nan"),
             avg_trade_return=float(tr["ret"].mean()) if len(tr) else float("nan"),
             avg_bars_held=float(tr["bars_held"].mean()) if len(tr) else float("nan"),
             turnover_annualized=float(res["traded"] / eq.mean() / years), total_traded_notional=float(res["traded"]),
             avg_gross_exposure=res["exposure"], blocked_actions=int(res["blocked"]),
             start=str(DATES[start].date()), end=str(DATES[end].date()), final_equity=float(eq[-1]))
    return m

def make_model_policy(model, cfg):
    def fn(t, pos, up, hold, PXbt):
        acts = np.zeros(N, dtype=int); conf = np.zeros(N)
        x = torch.from_numpy(PXbt[:, t - SEQ + 1:t + 1, :]).to(DEVICE)          # (N, L, F)
        with torch.no_grad():
            probs = torch.softmax(model.eval()(x), dim=-1).cpu().numpy()
        a, c = probs.argmax(1), probs.max(1)
        gate = (a >= 1) & (a <= 2) & (c < cfg.min_entry_confidence)
        a[gate] = 0
        return a, c
    return fn

def make_rule_policy(cfg):
    iz, ir, ia, iat = FIDX["zscore_20"], FIDX["rsi_14"], FIDX["adx_14"], FIDX["atr_14_pct"]
    def fn(t, pos, up, hold, PXbt):
        z, rsi, adx, atrp = PR[:, t, iz], PR[:, t, ir], PR[:, t, ia], PR[:, t, iat]
        a = np.zeros(N, dtype=int); flat = pos == 0
        a[flat & (z <= -cfg.z_entry) & (rsi <= cfg.rsi_oversold) & (adx <= cfg.adx_max)] = 1
        if cfg.allow_short:
            a[flat & (z >= cfg.z_entry) & (rsi >= cfg.rsi_overbought) & (adx <= cfg.adx_max)] = 2
        stop = up <= -cfg.stop_atr_mult * atrp
        a[(pos == 1) & ((z >= -cfg.exit_z) | stop | (hold >= cfg.max_hold_bars))] = 3
        a[(pos == -1) & ((z <= cfg.exit_z) | stop | (hold >= cfg.max_hold_bars))] = 3
        return a, np.ones(N)
    return fn

def bench_equity(start, end): return cfg.initial_capital * B[start:end + 1] / B[start]

BT = {}
for name, (s, e) in {"validation": (I_TRAIN_END, I_VAL_END - 1), "test": (I_VAL_END, T - 1)}.items():
    r_model = run_backtest(make_model_policy(model, cfg), s, e, cfg)
    r_rule = run_backtest(make_rule_policy(cfg), s, e, cfg)
    bq = bench_equity(s, e)
    BT[name] = dict(start=s, end=e, model=r_model, rule=r_rule, bench=bq,
                    metrics=dict(model=summarize(r_model, s, e, cfg), rule_baseline=summarize(r_rule, s, e, cfg),
                                 benchmark=dict(**perf(bq, cfg.risk_free_annual), start=str(DATES[s].date()), end=str(DATES[e].date()))))

cols = ["cumulative_return", "cagr", "annual_vol", "sharpe", "max_drawdown", "turnover_annualized", "trade_count", "win_rate", "avg_gross_exposure", "blocked_actions"]
for split in ["test", "validation"]:
    print(f"\n=== {split.upper()} ({BT[split]['metrics']['model']['start']} -> {BT[split]['metrics']['model']['end']}) ===")
    print(pd.DataFrame(BT[split]["metrics"]).T.reindex(columns=cols).round(4).to_string())
print("\nModel = learned policy | rule_baseline = same rule, no learning, no future filter | benchmark =", cfg.benchmark, "buy&hold")
print("Costs:", cfg.fee_bps, "bps fee +", cfg.slippage_bps, "bps slippage per side | allow_short =", cfg.allow_short)

## 13. Backtest plots

In [ ]:
bt = BT["test"]; s, e = bt["start"], bt["end"]; dts = DATES[s:e + 1]
fig, ax = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
for lab_, arr, kw in [("Mean-reversion policy (model)", bt["model"]["equity"], {}),
                      ("Rule baseline (no learning)", bt["rule"]["equity"], {"ls": "--"}),
                      (f"{cfg.benchmark} buy & hold", bt["bench"], {"color": "grey"})]:
    ax[0].plot(dts, arr, label=lab_, **kw)
    ax[1].plot(dts, arr / np.maximum.accumulate(arr) - 1, **kw)
ax[0].set_title("Test-period equity (after fees and slippage)"); ax[0].legend(); ax[1].set_title("Drawdown")
plt.tight_layout(); plt.show()

tr = bt["model"]["trades"]
if len(tr):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
    ax[0].hist(tr["ret"] * 100, bins=40); ax[0].set_title("Model trade returns (%)")
    tr.groupby("side")["pnl"].sum().plot(kind="bar", ax=ax[1], title="Total P&L by side ($)")
    plt.tight_layout(); plt.show()

# example: one ticker with model signals on the test period
tk0 = 0
x_all = np.stack([PX[tk0, t - SEQ + 1:t + 1] for t in range(I_VAL_END, T - 1)])
pb = torch.softmax(model.eval()(torch.from_numpy(x_all).to(DEVICE)), -1).detach().cpu().numpy()
# (illustration only: position input here is the rule-path state stored in the panel, not the backtest's live state)
sig = pb.argmax(1)
fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
d2 = DATES[I_VAL_END:T - 1]
ax[0].plot(d2, C[tk0, I_VAL_END:T - 1], lw=0.9); ax[0].plot(d2, SMA20[tk0, I_VAL_END:T - 1], lw=0.8, alpha=0.7)
for a_, mk, col in [(1, "^", "green"), (2, "v", "red"), (3, "x", "black")]:
    m_ = sig == a_; ax[0].scatter(d2[m_], C[tk0, I_VAL_END:T - 1][m_], marker=mk, c=col, s=22, label=ACTIONS[a_])
ax[0].set_title(f"{NAMES[tk0]}: close, SMA20 and predicted actions (rule-path state, illustration)"); ax[0].legend(ncol=3)
ax[1].plot(d2, PR[tk0, I_VAL_END:T - 1, FIDX["zscore_20"]], lw=0.8); ax[1].axhline(cfg.z_entry, ls=":"); ax[1].axhline(-cfg.z_entry, ls=":"); ax[1].set_title("z-score(20)")
plt.tight_layout(); plt.show()

## 14. `MeanReversionPolicy` class

Inference input (**common schema v1**, shared by all five policies): a raw (unscaled) feature window of shape `(seq_len, 29)` whose columns follow `feature_schema.json` — as a `DataFrame` (columns matched by name), a NumPy array in schema order, or a dict `{feature_name: sequence}`. A leading batch dimension is also accepted. Missing **profile** columns are filled with the configured defaults; missing market/state columns raise an error. The preprocessor is applied inside `predict`.

`predict(state)` returns `probabilities` (4), `action`, `action_name`, `confidence` (= max probability) and the list of currently `valid_actions`.

In [ ]:
class MeanReversionPolicy:
    NAME = "SP-02 Mean-Reversion Policy"
    FORMAT_VERSION = 1

    def __init__(self, model, preprocessor, config, schema, device="cpu"):
        self.device = torch.device(device)
        self.model = model.to(self.device).eval()
        self.pre, self.config, self.schema = preprocessor, config, schema
        self.feature_names = list(schema["feature_names"]); self.seq_len = int(schema["seq_len"])

    # ---- input handling ----
    def _to_array(self, state):
        names, L = self.feature_names, self.seq_len
        if isinstance(state, dict) and "features" in state: state = state["features"]
        if isinstance(state, dict): state = pd.DataFrame(state)
        if isinstance(state, pd.DataFrame):
            df = state.copy()
            for k, v in self.config["profile"].items():
                if k not in df.columns: df[k] = v
            missing = [n for n in names if n not in df.columns]
            if missing: raise ValueError(f"missing features: {missing}")
            arr = df[names].to_numpy(dtype=np.float64)
        else:
            arr = np.asarray(state, dtype=np.float64)
        if arr.ndim not in (2, 3) or arr.shape[-1] != len(names):
            raise ValueError(f"expected (..., seq_len, {len(names)}) raw features, got {arr.shape}")
        if arr.shape[-2] < L: raise ValueError(f"need at least {L} timesteps, got {arr.shape[-2]}")
        arr = arr[..., -L:, :]
        if not np.isfinite(arr).all(): raise ValueError("non-finite values in input")
        return arr

    @torch.no_grad()
    def predict(self, state):
        arr = self._to_array(state); single = arr.ndim == 2
        x = torch.from_numpy(self.pre.transform(arr[None] if single else arr)).to(self.device)
        valid = self.model.valid_mask(x).cpu().numpy()
        probs = torch.softmax(self.model(x), dim=-1).cpu().numpy()
        act, conf = probs.argmax(1), probs.max(1)
        def pack(i):
            return {"probabilities": [float(p) for p in probs[i]],
                    "probabilities_by_action": {ACTIONS[k]: float(probs[i, k]) for k in range(4)},
                    "action": int(act[i]), "action_name": ACTIONS[int(act[i])], "confidence": float(conf[i]),
                    "valid_actions": [ACTIONS[k] for k in range(4) if valid[i, k]]}
        return pack(0) if single else [pack(i) for i in range(len(act))]

    # ---- persistence (tensors + plain python only -> safe for torch.load(weights_only=True)) ----
    def save(self, path):
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        torch.save({"format_version": self.FORMAT_VERSION, "name": self.NAME,
                    "model_state_dict": {k: v.detach().cpu() for k, v in self.model.state_dict().items()},
                    "model_kwargs": MODEL_KWARGS_JSON, "config": self.config, "schema": self.schema,
                    "preprocessor": self.pre.to_dict()}, path)

    @classmethod
    def load(cls, path, device="cpu"):
        ck = torch.load(path, map_location=device, weights_only=True)
        kw = ck["model_kwargs"]
        model = MeanReversionNet(**kw); model.load_state_dict(ck["model_state_dict"])
        return cls(model, Preprocessor.from_dict(ck["preprocessor"]), ck["config"], ck["schema"], device)

MODEL_KWARGS_JSON = {k: (v if not isinstance(v, np.generic) else v.item()) for k, v in MODEL_KWARGS.items()}
print("class defined")

## 15. Export artifacts

| File | Content |
|---|---|
| `mean_reversion_policy.pt` | weights + model kwargs + config + schema + preprocessor stats (self-contained) |
| `mean_reversion_config.json` | full configuration, split dates, label definition, training summary |
| `feature_schema.json` | ordered input schema (common schema v1), action map, I/O spec |
| `preprocessing.pkl` | fitted sklearn `StandardScaler` + preprocessor parameters (pickle: load only in trusted environments) |
| `evaluation_metrics.json` | classification metrics (val/test) and backtest metrics (val/test) with disclosures |
| `equity_curve.csv` | test-period equity: model, rule baseline, benchmark |

In [ ]:
AD = cfg.artifact_dir
cfg_dict = json.loads(json.dumps(asdict(cfg), default=str))

schema = {
    "schema_name": "trading_agent_common_policy_schema", "schema_version": "1.0",
    "policy": "SP-02 Mean-Reversion Policy",
    "seq_len": int(cfg.seq_len), "n_features": int(F), "feature_names": FEATURE_NAMES,
    "feature_groups": {"market": MARKET_FEATURES, "position_state": STATE_FEATURES, "trader_profile": PROFILE_FEATURES},
    "feature_details": {
        "input_is_raw_unscaled": True,
        "position": "-1 short, 0 flat, +1 long; position held at the close of the decision bar",
        "unrealized_pnl": "position * (close / entry_fill_price - 1), fraction",
        "holding_bars": "bars since entry fill, counting the entry bar (entry bar = 1)",
        "returns": "log returns", "rsi_14": "Wilder, 0..100", "adx_14": "Wilder, 0..100",
        "profile_defaults": cfg.profile, "profile_supplied_in_training": cfg.profile_supplied,
        "market_scaling": "StandardScaler fit on train period only, clipped to +-5",
        "state_scaling": {"unrealized_pnl": "x / 0.10 clipped +-5", "holding_bars": f"x / {cfg.max_hold_bars} clipped to [0,3]"},
    },
    "mean_reversion_emphasis_features": ["zscore_20", "zscore_60", "sma_dev_20", "bb_pos_20", "bb_width_20", "rsi_14", "eff_ratio_20", "vol_ratio_20", "adx_14"],
    "action_map": {str(k): v for k, v in ACTIONS.items()},
    "action_semantics": {"BUY": "open long (flat only)", "SELL": "open short (flat only, requires allow_short)", "EXIT": "close any open position", "HOLD": "no change"},
    "validity_mask": "BUY/SELL invalid unless position==0; SELL invalid if allow_short is false; EXIT invalid if position==0; enforced inside the network",
    "decision_timing": "decision at close of bar t, execution at open of bar t+1",
    "output": {"probabilities": "float[4] ordered by action id", "action": "int 0..3", "confidence": "max probability"},
}
schema["schema_sha256"] = hashlib.sha256(json.dumps({k: schema[k] for k in ["seq_len", "feature_names", "action_map"]}, sort_keys=True).encode()).hexdigest()

policy = MeanReversionPolicy(model.cpu(), PRE, cfg_dict, schema, device="cpu")
model.to(DEVICE)
P_PT = os.path.join(AD, "mean_reversion_policy.pt")
policy.save(P_PT)

config_out = {"policy": "SP-02 Mean-Reversion Policy", "config": cfg_dict, "label_source": cfg.label_source, "label_disclosure": LABEL_DISCLOSURE,
              "label_definition": {"z_entry": cfg.z_entry, "rsi_oversold": cfg.rsi_oversold, "rsi_overbought": cfg.rsi_overbought, "adx_max": cfg.adx_max,
                                   "label_horizon_bars": cfg.label_horizon, "reversion_frac": cfg.reversion_frac, "stop_atr_mult": cfg.stop_atr_mult,
                                   "exit_z": cfg.exit_z, "max_hold_bars": cfg.max_hold_bars},
              "universe": NAMES, "benchmark": cfg.benchmark,
              "splits": {"train": [str(DATES[0].date()), str(DATES[I_TRAIN_END - 1].date())], "validation": [str(DATES[I_TRAIN_END].date()), str(DATES[I_VAL_END - 1].date())],
                         "test": [str(DATES[I_VAL_END].date()), str(DATES[-1].date())], "purge_bars": PURGE},
              "training": {"best_epoch": BEST_EPOCH, "epochs_run": len(hist["train_loss"]), "best_val_loss": float(best), "class_weights": W.tolist(),
                          "train_samples": int(len(y_tr)), "val_samples": int(len(y_va)), "test_samples": int(len(y_te))},
              "environment": {"torch": torch.__version__, "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__, "python": platform.python_version()}}
json.dump(config_out, open(os.path.join(AD, "mean_reversion_config.json"), "w"), indent=2)
json.dump(schema, open(os.path.join(AD, "feature_schema.json"), "w"), indent=2)
pickle.dump({"preprocessor_params": PRE.to_dict(), "sklearn_scaler": PRE.scaler, "sklearn_version": sklearn.__version__,
             "feature_names": FEATURE_NAMES}, open(os.path.join(AD, "preprocessing.pkl"), "wb"))

metrics_out = {"disclosure": LABEL_DISCLOSURE,
               "caveats": ["Research prototype for backtesting/paper trading only.", "Classification metrics are teacher-forced against labels, not profitability.",
                           "Universe is a current large-cap list: survivorship bias.", "Daily bars; simple fee/slippage model; no liquidity constraints."],
               "classification": {"validation": cls_val, "test": cls_test},
               "backtest": {k: BT[k]["metrics"] for k in BT},
               "backtest_settings": {"fee_bps": cfg.fee_bps, "slippage_bps": cfg.slippage_bps, "borrow_bps_annual": cfg.borrow_bps_annual, "allow_short": cfg.allow_short,
                                     "max_position_frac": cfg.max_position_frac, "max_gross_exposure": cfg.max_gross_exposure, "execution": "next-bar open"}}
json.dump(metrics_out, open(os.path.join(AD, "evaluation_metrics.json"), "w"), indent=2, default=float)

eqdf = pd.DataFrame({"date": DATES[BT["test"]["start"]:BT["test"]["end"] + 1], "model_equity": BT["test"]["model"]["equity"],
                     "rule_baseline_equity": BT["test"]["rule"]["equity"], "benchmark_equity": BT["test"]["bench"]})
eqdf.to_csv(os.path.join(AD, "equity_curve.csv"), index=False)
for f_ in sorted(os.listdir(AD)): print(f"{os.path.join(AD, f_):45s} {os.path.getsize(os.path.join(AD, f_)):>10,d} bytes")

## 16. Inference demonstration

Loads the saved `.pt` into a fresh object, predicts on a real raw feature window from the test period, checks it matches the in-memory policy, and shows the validity mask in action (forcing `position=+1` removes BUY/SELL probability).

In [ ]:
loaded = MeanReversionPolicy.load(P_PT, device="cpu")
n_demo, t_demo = int(te_tk[len(te_tk) // 2]), int(te_t[len(te_t) // 2])
window = pd.DataFrame(PR[n_demo, t_demo - SEQ + 1:t_demo + 1], columns=FEATURE_NAMES, index=DATES[t_demo - SEQ + 1:t_demo + 1])
print(f"window: {NAMES[n_demo]} ending {DATES[t_demo].date()} | shape {window.shape}")
out = loaded.predict(window)
print(json.dumps(out, indent=2))

ref = policy.predict(window)
print("save/load max |dp|:", max(abs(a - b) for a, b in zip(out["probabilities"], ref["probabilities"])))

w_long = window.copy(); w_long.loc[w_long.index[-1], "position"] = 1.0
print("\nforced position=+1 ->", loaded.predict(w_long)["probabilities_by_action"], "| valid:", loaded.predict(w_long)["valid_actions"])

# dict-of-sequences input, profile columns omitted (filled with defaults)
as_dict = {c: window[c].tolist() for c in MARKET_FEATURES + STATE_FEATURES}
print("\ndict input (no profile) ->", loaded.predict(as_dict)["action_name"])

## 17. Meta-RL compatibility report

Programmatic checks on this policy, plus a helper to diff `feature_schema.json` against the Momentum Policy's (or any other policy's) schema. **The Momentum Policy's schema was not provided to this notebook**, so cross-policy identity cannot be verified here; this notebook defines *common schema v1* and the diff helper tells you exactly what differs.

In [ ]:
checks = []
def chk(name, ok, detail=""): checks.append((name, bool(ok), detail))

chk("action map is {0:HOLD,1:BUY,2:SELL,3:EXIT}", ACTIONS == {0: "HOLD", 1: "BUY", 2: "SELL", 3: "EXIT"})
bt_probs = loaded.predict(np.stack([PR[te_tk[i], te_t[i] - SEQ + 1:te_t[i] + 1] for i in range(0, len(te_tk), max(1, len(te_tk) // 500))]))
sums = np.array([sum(o["probabilities"]) for o in bt_probs])
chk("output has 4 probabilities summing to 1", all(len(o["probabilities"]) == 4 for o in bt_probs) and np.allclose(sums, 1, atol=1e-5), f"max |sum-1| = {np.abs(sums-1).max():.2e}")
chk("action == argmax and confidence == max prob", all(o["action"] == int(np.argmax(o["probabilities"])) and abs(o["confidence"] - max(o["probabilities"])) < 1e-9 for o in bt_probs))
inv = max(sum(p for p, nm in zip(o["probabilities"], ACTIONS.values()) if nm not in o["valid_actions"]) for o in bt_probs)
chk("zero probability on invalid actions", inv < 1e-6, f"max invalid mass = {inv:.2e}")
chk("input width == schema n_features == model input", loaded.model.encoder.input_size == schema["n_features"] == len(schema["feature_names"]) == F)
chk("save/load round trip identical", max(abs(a - b) for a, b in zip(out["probabilities"], ref["probabilities"])) < 1e-6)
chk("deterministic inference", loaded.predict(window)["probabilities"] == loaded.predict(window)["probabilities"])
chk("all six artifacts exist", all(os.path.exists(os.path.join(AD, f_)) for f_ in ["mean_reversion_policy.pt", "mean_reversion_config.json", "feature_schema.json", "preprocessing.pkl", "evaluation_metrics.json", "equity_curve.csv"]))
chk("model never proposed an invalid action in the backtest", BT["test"]["metrics"]["model"]["blocked_actions"] == 0 and BT["validation"]["metrics"]["model"]["blocked_actions"] == 0, "blocked: test=%d, val=%d" % (BT["test"]["metrics"]["model"]["blocked_actions"], BT["validation"]["metrics"]["model"]["blocked_actions"]))
chk("labels are proxy (strategy imitation)" if cfg.label_source.startswith("proxy") else "labels are genuine trader actions", True, cfg.label_source)

for n_, ok_, d_ in checks:
    print(("PASS " if ok_ else "FAIL ") + n_ + (f"  [{d_}]" if d_ else ""))

def compare_schema(other_path, mine_path=os.path.join(AD, "feature_schema.json")):
    a, b = json.load(open(mine_path)), json.load(open(other_path))
    rep = {"same_seq_len": a["seq_len"] == b["seq_len"], "same_action_map": a["action_map"] == b["action_map"],
           "same_feature_order": a["feature_names"] == b["feature_names"],
           "only_in_this": [f for f in a["feature_names"] if f not in b["feature_names"]],
           "only_in_other": [f for f in b["feature_names"] if f not in a["feature_names"]]}
    rep["identical_input_schema"] = rep["same_seq_len"] and rep["same_action_map"] and rep["same_feature_order"]
    return rep
# usage: compare_schema("/path/to/momentum/feature_schema.json")

print("\nschema_sha256:", schema["schema_sha256"])
print("""
META-RL COMPATIBILITY NOTES
 * Output contract: probabilities[4] (HOLD,BUY,SELL,EXIT), action, confidence=max prob, valid_actions.
 * State inputs the Meta-RL must supply per step: raw window (seq_len x 29) in schema order, incl. position / unrealized_pnl / holding_bars.
 * Policy-specific features (zscore_*, bb_*, rsi, eff_ratio, ...) are part of the 22 market features. Recommended: use this 29-feature
   schema as the shared superset for all five policies (each policy may ignore columns it does not need).
 * NOT verifiable here: that the Momentum Policy uses exactly this schema (not provided). Run compare_schema() against its feature_schema.json.
 * Confidence is a softmax max-probability, NOT a calibrated probability of profit. Calibrate on validation data before using it for Meta-RL weighting.
 * Trader-profile inputs were constants in training; the policy is not personalized until trained with real profile data.
 * Position state must be maintained by the caller using the same definitions (see feature_schema.json).
""")

## 18. Artifact paths and download

In [ ]:
print("ARTIFACT PATHS")
for f_ in ["mean_reversion_policy.pt", "mean_reversion_config.json", "feature_schema.json", "preprocessing.pkl", "evaluation_metrics.json", "equity_curve.csv"]:
    print("  ", os.path.abspath(os.path.join(AD, f_)))
print("\n" + LABEL_DISCLOSURE)
print("Research prototype for backtesting and paper trading only.")

try:                                   # Colab convenience: zip and download
    import shutil
    from google.colab import files
    shutil.make_archive("sp02_mean_reversion_artifacts", "zip", AD)
    files.download("sp02_mean_reversion_artifacts.zip")
except Exception as e:
    print("(not in Colab or download skipped:", type(e).__name__, ")")